# Columns, slab and contacts

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps
%pip install -q compas_model==0.9.3

In [ ]:
from compas.geometry import Polygon, Translation
from compas_model.elements import ColumnElement, PlateElement
from compas_model.models import Model
from compas_notebook import Viewer
from compas_notebook.config import Config

## Four columns and a slab

In [ ]:
model = Model(name="Pavilion")
for x, y in [(0, 0), (4, 0), (4, 3), (0, 3)]:
    model.add_element(ColumnElement(width=0.3, depth=0.3, height=3.0, transformation=Translation.from_vector([x, y, 0])))
slab = PlateElement(Polygon([[-0.5, -0.5, 3], [-0.5, 3.5, 3], [4.5, 3.5, 3], [4.5, -0.5, 3]]), thickness=0.2)
model.add_element(slab)
slab.modelgeometry.flip_cycles()

## Contacts

In [ ]:
model.compute_contacts(tolerance=1e-3, minimum_area=1e-3)
contacts = list(model.contacts())
[contact.size for contact in contacts]

## Slab weight on each contact [kN/m²]

In [ ]:
weight = slab.modelgeometry.volume() * 470 * 9.81 / 1000
[weight / 4 / contact.size for contact in contacts]

## Embodied carbon [kg CO₂e], example factors

In [ ]:
density = {"ColumnElement": 2400, "PlateElement": 470}
gwp = {"ColumnElement": 0.10, "PlateElement": 0.30}
carbon = {}
for element in model.elements():
    kind = type(element).__name__
    carbon[kind] = carbon.get(kind, 0) + element.modelgeometry.volume() * density[kind] * gwp[kind]
carbon

## Scene

In [ ]:
config = Config()
config.view.width, config.view.height = 700, 360
config.view.camera.position = [9, -7, 6]
config.view.camera.target = [2, 1.5, 1.5]
viewer = Viewer(config=config)
for element in model.elements():
    viewer.scene.add(element.modelgeometry, color="#cccccc", opacity=0.5, show_edges=True)
for contact in contacts:
    viewer.scene.add(contact.polygon, color="#dc4b8d")
viewer.show()